In [61]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "hanus2011comparing")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Floating-Peanut_All_Data_exp2-4all.csv")
complete_path_2 = os.path.join(original_data_pathway, "Floating-Peanut_All_Data_exp3_drink.csv")
# complete_path_3 = os.path.join(original_data_pathway, "Floating-Peanut_All_Data_exp3_loc.csv")
complete_path_4 = os.path.join(original_data_pathway, "exp_1.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [62]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_csv(complete_path_1,)
df1[['day','month', 'year']] = df1['date'].str.split('.',expand=True)
df1 = df1[~df1.species.str.contains("human")]
df1['experiment']='2'
df1['subject'] = df1['subject'].str.rstrip()
df1['species'] = df1['species'].str.rstrip()

species_rename = [['orang','orangutan'],['chimp','chimpanzee']]
for x,y in species_rename:
    df1['species'].replace(x, y, inplace=True, regex=True)
# df1['species'].unique()

In [ ]:
df2 = pd.read_csv(complete_path_2)
df2['subject'] = df2['subject'].str.rstrip()
df2['experiment']='3'
df2['subject'].unique()

remove_list_1 = [] ##names removed 
df2 = df2[~df2.subject.isin(remove_list_1)]

In [64]:
df2_temp = df2[['subject', 'drinker','experiment']]
df2_temp = df2_temp.assign(spit='no')
df2_temp = df2_temp.assign(success='no')
df2_spit_success_list = [['Tai','yes','yes'],
                         ['Ulla','yes','no'],
                         ['Fifi','yes','no'],
                         ['Jahaga','yes','no'],
                         ['Lome','yes','yes']]

for x,y,k in df2_spit_success_list:
    df2_temp.loc[df2_temp.subject == x, ['spit', 'success']] = y,k

# df2_temp['subject'].unique()

In [ ]:
# df3 = pd.read_csv(complete_path_3)
# df3['subject'] = df3['subject'].str.rstrip()
# df3['experiment']='3'
# df3['subject'].unique()

# remove_list_2 = []##names removed 

# df3 = df3[~df3.subject.isin(remove_list_2)]

In [66]:
df4 = pd.read_csv(complete_path_4)
df4['experiment']='1'

In [67]:
data_frames=[df1, df2_temp, df4]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x.rename(columns={"subject": "participant",
                      "age":"age_original",
                        "species":'species_original',
                        "cond":'condition',
                        "sex":"sex_original",
                        "species":"species_original"}, inplace=True) ##standardize names for participants
    x['participant'] = x['participant'].str.rstrip() ##remove spaces
    x['study_id']="hanus2011comparing"
    data_frames[index]=x
new_df=data_frames[0]
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)


In [68]:
comp_path_name_errors = os.path.join(pathway_gen, "hanus_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['participant'] = fulldf['participant'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['participant'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(original_data_pathway, "table_1.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf= fulldf.merge(apedf,left_on='participant', right_on='name', how='left')


period_remove = ['trial','latency_first_spit/water','latency_success']
for x in period_remove:
    fulldf[x].replace('.', '', inplace=True)
fulldf=fulldf.sort_values(by = ['participant','species'])


In [69]:
fulldf = fulldf[['study_id', 'experiment','year',  'month','day','participant','age_original','age_in_years',
                'sex','species', 
       'trial', 'condition', 'spitting',  
        'drinker',
       'latency_first_spit/water','spit', 'number_of_spits', 'success',
       'latency_success']]


In [70]:

for index in range(1,4):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'hanus2011comparing_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'hanus2011comparing_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
